In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    FunctionTransformer,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    auc,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin


In [3]:
df = pd.read_csv('data/training.csv')
df

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,12/2/2009,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,12/2/2009,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,12/2/2009,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,12/2/2009,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [4]:
def time_split(df, date_field, val_date, test_date):
    dates = pd.to_datetime(df[date_field])
    v_date = pd.to_datetime(val_date)
    t_date = pd.to_datetime(test_date)
    
    return (
        df[dates < v_date],
        df[(dates >= v_date) & (dates < t_date)],
        df[dates >= t_date]
    )

dt = pd.to_datetime(df["PurchDate"])
unique_dates = pd.Index(dt.dropna().sort_values().unique())

idx1 = len(unique_dates) // 3
idx2 = 2 * len(unique_dates) // 3

validation_date = unique_dates[idx1]
test_date = unique_dates[idx2]

train_df, valid_df, test_df = time_split(
    df=df,
    date_field="PurchDate",
    val_date=validation_date,
    test_date=test_date
)


In [5]:
df_train = train_df.copy()
df_valid = valid_df.copy()

In [6]:
target = 'IsBadBuy'
date = 'PurchDate'

cat_features = train_df.select_dtypes(include="object").columns
num_features = valid_df.select_dtypes(exclude="object").drop(columns=[target]).columns
cat_features = cat_features.drop("PurchDate", errors="ignore")

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

X_train = preprocessor.fit_transform(train_df)
X_valid = preprocessor.transform(valid_df)

y_train = train_df[target]
y_valid = valid_df[target]

In [8]:
lr = LogisticRegression(max_iter=1000)
nb = GaussianNB()
knn = KNeighborsClassifier(n_neighbors=10)

In [9]:
X_train_d = X_train.toarray()
X_valid_d = X_valid.toarray()

lr.fit(X_train, y_train)
nb.fit(X_train_d, y_train)
knn.fit(X_train, y_train)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [10]:
def gini_sklearn(y_true, y_pred_proba):
    return abs(2 * roc_auc_score(y_true, y_pred_proba) - 1)

In [10]:
gini_lr = gini_sklearn(
    y_valid,
    lr.predict_proba(X_valid)[:, 1]
)

gini_nb = gini_sklearn(
    y_valid,
    nb.predict_proba(X_valid_d)[:, 1]
)

gini_knn = gini_sklearn(
    y_valid,
    knn.predict_proba(X_valid)[:, 1]
)

print("Logistic Regression Gini:", gini_lr)
print("Naive Bayes Gini:", gini_nb)
print("KNN Gini:", gini_knn)


Logistic Regression Gini: 0.31618832596820745
Naive Bayes Gini: 0.052073940954982634
KNN Gini: 0.21482479667202337


Самая лучшая модель - это логистическая регрессия. Потому что в датасете признаки или категориальные или закодированные через OneHotEncoding, из-за этого данные становятся высокоразмерными и хорошо делятся линейно, а лин. рег. эффективно находит гиперплоскость разделения. Для Наивного Байесовского алгоритмма нужно, чтобы признаки были независимые. Это здесь не выполняется. Для KNN после кодирования соседи становятся далекими и расстояние между ними не информативно, так же он чувствителен к шуму. 

## 5. Implement Gini score calculation.

In [ ]:
def my_roc_auc_score(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    n = len(y_true)
    n_pos = int(y_true.sum())
    n_neg = n - n_pos

    if n_pos == 0 or n_neg == 0:
        return 0.5

    order = np.argsort(y_score)
    scores_sorted = y_score[order]
    y_sorted = y_true[order]

    ranks = np.empty(n, dtype=float)

    i = 0
    rank = 1
    while i < n:
        j = i
        while j + 1 < n and scores_sorted[j + 1] == scores_sorted[i]:
            j += 1

        avg_rank = (rank + (rank + (j - i))) / 2.0
        ranks[i : j + 1] = avg_rank

        rank += j - i + 1
        i = j + 1

    sum_ranks_pos = ranks[y_sorted == 1].sum()
    u_pos = sum_ranks_pos - (n_pos * (n_pos + 1)) / 2.0
    auc = u_pos / (n_pos * n_neg)

    return float(auc)

def my_gini_score(y_true, y_score):
    auc = my_roc_auc_score(y_true, y_score)
    return abs(2 * auc - 1)

In [12]:
my_gini_lr = my_gini_score(
    y_valid,
    lr.predict_proba(X_valid)[:, 1]
)

my_gini_nb = my_gini_score(
    y_valid,
    nb.predict_proba(X_valid_d)[:, 1]
)

my_gini_knn = my_gini_score(
    y_valid,
    knn.predict_proba(X_valid)[:, 1]
)

print("Logistic Regression MyGini:", my_gini_lr)
print("Naive Bayes MyGini:", my_gini_nb)
print("KNN MyGini:", my_gini_knn)


Logistic Regression MyGini: 0.31618832596820745
Naive Bayes MyGini: 0.052073940954982634
KNN MyGini: 0.21482479667202337


## 6. Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers.

In [12]:
class MyLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__(self, lr=0.01, n_epochs=10, batch_size=1, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.weights = None
        self.intercept = None

    def _sigmoid(self, z):
        z = np.clip(z, -35, 35)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = X.values if isinstance(X, pd.DataFrame) else X
        y = y.values if isinstance(y, (pd.Series, pd.DataFrame)) else y
        y = y.ravel()

        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)
        self.weights = np.zeros(n_features)
        self.intercept = 0.0

        for _ in range(self.n_epochs):
            indices = rng.permutation(n_samples)
            for start in range(0, n_samples, self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                Xb = X[batch_idx]
                yb = y[batch_idx]
                linear = Xb @ self.weights + self.intercept
                pred = self._sigmoid(linear)
                diff = pred - yb
                self.weights -= self.lr * (Xb.T @ diff) / len(batch_idx)
                self.intercept -= self.lr * np.mean(diff)

        return self

    def predict_proba(self, X):
        X = X.values if isinstance(X, pd.DataFrame) else X
        p = self._sigmoid(X @ self.weights + self.intercept)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [ ]:
from scipy.sparse import issparse
class MyGaussianNB(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.class_probs = {}
        self.class_means = {}
        self.class_std = {}
        self.classes = None

    def fit(self, X, y):
        if issparse(X):
            X = X.toarray()
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).ravel()
        self.classes = np.unique(y)
        n_samples, n_features = X.shape

        for cls in self.classes:
            Xc = X[y == cls]
            self.class_probs[cls] = Xc.shape[0] / n_samples
            self.class_means[cls] = Xc.mean(axis=0)
            self.class_std[cls] = np.std(Xc, axis=0) + 1e-9

        return self

    def _log_gaussian_pdf(self, X, mean, std):
        return -0.5 * np.log(2 * np.pi * std**2) - 0.5 * ((X - mean) ** 2) / (std**2)

    def predict_proba(self, X):
        if issparse(X):
            X = X.toarray()
        X = np.asarray(X, dtype=float)
        log_probs = []

        for cls in self.classes:
            mean = self.class_means[cls]
            std = self.class_std[cls]
            log_prob = np.log(self.class_probs[cls]) + np.sum(self._log_gaussian_pdf(X, mean, std), axis=1)
            log_probs.append(log_prob)

        log_probs = np.stack(log_probs, axis=1)
        log_probs -= log_probs.max(axis=1, keepdims=True)
        probs = np.exp(log_probs)
        probs /= probs.sum(axis=1, keepdims=True)

        return probs

    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes[np.argmax(probs, axis=1)]


In [14]:
class MyKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5, batch_size=128):
        self.n_neighbors = n_neighbors
        self.batch_size = batch_size
        self.X_train = None
        self.y_train = None
        self.train_norm2 = None

    def fit(self, X, y):
        if issparse(X):
            X = X.toarray()
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y).ravel()
        self.train_norm2 = np.sum(self.X_train**2, axis=1)
        return self

    def predict_proba(self, X):
        if self.X_train is None:
            raise RuntimeError("Model is not fitted yet.")
        if issparse(X):
            X = X.toarray()
        X = np.asarray(X, dtype=float)
        n_samples = X.shape[0]
        probs = []

        for start in range(0, n_samples, self.batch_size):
            end = min(start + self.batch_size, n_samples)
            Xb = X[start:end]
            q_norm2 = np.sum(Xb**2, axis=1)
            dots = Xb @ self.X_train.T
            dist2 = q_norm2[:, None] + self.train_norm2[None, :] - 2 * dots
            dist2 = np.maximum(dist2, 0.0)

            batch_probs = []
            for i in range(dist2.shape[0]):
                nn_idx = np.argpartition(dist2[i], self.n_neighbors - 1)[:self.n_neighbors]
                counts = {}
                for cls in np.unique(self.y_train):
                    counts[cls] = np.sum(self.y_train[nn_idx] == cls) / self.n_neighbors
                batch_probs.append([counts[cls] for cls in sorted(counts.keys())])
            probs.extend(batch_probs)

        return np.array(probs)

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

In [16]:
lr = MyLogisticRegression()

lr.fit(X_train, y_train)
y_pred_proba = lr.predict_proba(X_valid)[:, 1]
my_gini_my_lr = my_gini_score(y_valid, y_pred_proba)

my_gini_my_lr

0.33310260972953043

In [17]:
nb = MyGaussianNB()
nb.fit(X_train, y_train)
y_pred_proba = nb.predict_proba(X_valid)[:, 1]

gini_gauss = 2 * my_roc_auc_score(y_valid, y_pred_proba) - 1
print("Gini Naive:", gini_gauss)

Gini Naive: 0.052073940954982634


In [18]:
knnmy = MyKNN()
knnmy.fit(X_train, y_train)
y_pred_proba = knnmy.predict_proba(X_valid)[:, 1]

gini_knn = 2 * my_roc_auc_score(y_valid, y_pred_proba) - 1
print("Gini KNN:", gini_knn)

Gini KNN: 0.17251705455373423


## 7. Try to create non-linear features

In [15]:
def add_non_linear_features(df, reference_df):
    df = df.copy()
    eps = 1e-6

    df["odo_per_age"] = df["VehOdo"] / (df["VehicleAge"] + 1)
    df["cost_to_retail"] = df["VehBCost"] / (df["MMRCurrentRetailAveragePrice"] + eps)
    df["warranty_to_cost"] = df["WarrantyCost"] / (df["VehBCost"] + eps)

    make_mean_cost = reference_df.groupby("Make")["VehBCost"].mean()
    auction_mean_cost = reference_df.groupby("Auction")["VehBCost"].mean()

    df["make_mean_cost"] = df["Make"].map(make_mean_cost)
    df["auction_mean_cost"] = df["Auction"].map(auction_mean_cost)

    return df


In [16]:
train_df_nf = add_non_linear_features(df_train, df_train)
valid_df_nf = add_non_linear_features(df_valid, df_train)
cat_features_nf = train_df_nf.select_dtypes(include="object").columns
cat_features_nf = cat_features_nf.drop("PurchDate", errors="ignore")
num_feature_nf = train_df_nf.select_dtypes(exclude="object").drop(columns=[target]).columns
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_nf = ColumnTransformer([
    ("num", num_pipeline, num_feature_nf),
    ("cat", cat_pipeline, cat_features_nf)
])


X_train_nf = preprocessor_nf.fit_transform(train_df_nf)
X_valid_nf = preprocessor_nf.transform(valid_df_nf)

y_train_nf = train_df_nf[target]
y_valid_nf = valid_df_nf[target]
num_feature_nf

Index(['RefId', 'VehYear', 'VehicleAge', 'WheelTypeID', 'VehOdo',
       'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice', 'BYRNO',
       'VNZIP1', 'VehBCost', 'IsOnlineSale', 'WarrantyCost', 'odo_per_age',
       'cost_to_retail', 'warranty_to_cost', 'make_mean_cost',
       'auction_mean_cost'],
      dtype='object')

In [17]:
lr = LogisticRegression(max_iter=1000)
nb= GaussianNB()
knn = KNeighborsClassifier(n_neighbors=10)

In [18]:
lr.fit(X_train, y_train)
nb.fit(X_train_d, y_train)
knn.fit(X_train, y_train)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [19]:
gini_lr = gini_sklearn(
    y_valid,
    lr.predict_proba(X_valid)[:, 1]
)

gini_nb = gini_sklearn(
    y_valid,
    nb.predict_proba(X_valid_d)[:, 1]
)

gini_knn = gini_sklearn(
    y_valid,
    knn.predict_proba(X_valid)[:, 1]
)

print("Logistic Regression Gini:", gini_lr)
print("Naive Bayes Gini:", gini_nb)
print("KNN Gini:", gini_knn)

Logistic Regression Gini: 0.31618832596820745
Naive Bayes Gini: 0.052073940954982634
KNN Gini: 0.21482479667202337


In [20]:
X_train_nf_d = X_train_nf.toarray()
X_valid_nf_d = X_valid_nf.toarray()
lr_nl = LogisticRegression(max_iter=1000)
nb_nl= GaussianNB()
knn_nl = KNeighborsClassifier(n_neighbors=10)
lr_nl.fit(X_train_nf, y_train_nf)
nb_nl.fit(X_train_nf_d, y_train_nf)
knn_nl.fit(X_train_nf, y_train_nf)

,n_neighbors,10
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [ ]:
gini_lr_nf = gini_sklearn(
    y_valid_nf,
    lr_nl.predict_proba(X_valid_nf)[:, 1]
)

gini_nb_nf = gini_sklearn(
    y_valid_nf,
    nb_nl.predict_proba(X_valid_nf_d)[:, 1]
)

gini_knn_nf = gini_sklearn(
    y_valid_nf,
    knn_nl.predict_proba(X_valid_nf)[:, 1]
)

print("Logistic Regression Gini NF:", gini_lr_nf)
print("Naive Bayes Gini NF:", gini_nb_nf)
print("KNN Gini NF:", gini_knn_nf)

Logistic Regression Gini NF: 0.3175796284352157
Naive Bayes Gini NF: 0.052073940954982634
KNN Gini NF: 0.20934904709553326


Коэффицент gini в лог рег улучшился, в байесе такой же, в кнн упал

## 8. Feature selection

In [ ]:
coef = lr_nl.coef_[0]

feature_names = preprocessor_nf.get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef,
    "abs_coef": np.abs(coef)
}).sort_values("abs_coef", ascending=False)

coef_df.head(20)

,feature,coef,abs_coef
15,num__VehBCost,-2.143130,2.143130
4,num__VehOdo,1.536824,1.536824
1546,cat__SubModel_QUAD CAB 3.9L SLT,1.277645,1.277645
1520,cat__SubModel_PASSENGER 3.4L VALUE,1.253568,1.253568
141,cat__Model_AVEO 1.6L I4 DOHC,1.192307,1.192307
256,cat__Model_DURANGO 4WD V8 5.7L,-1.138933,1.138933
860,cat__Trim_C,1.136000,1.136000
1135,cat__SubModel_4D SEDAN HEMI,1.136000,1.136000
1610,cat__SubModel_REG CAB 5.4L XLT,1.133353,1.133353
523,cat__Model_MONTANA 3.4L V6 SFI,-1.120672,1.120672


In [ ]:
threshold = coef_df["abs_coef"].quantile(0.25)

mask = coef_df["abs_coef"] > threshold
mask.sum(), X_train_d.shape[1]
X_train_manual = X_train_nf[:, mask.values]
X_valid_manual = X_valid_nf[:, mask.values]


In [ ]:
lr_manual = LogisticRegression(max_iter=2000)
lr_manual.fit(X_train_manual, y_train_nf)

gini_manual = gini_sklearn(
    y_valid_nf,
    lr_manual.predict_proba(X_valid_manual)[:, 1]
)

print("Manual feature selection Gini:", gini_manual)

Manual feature selection Gini: 0.3129439750779075


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

lr_l1 = LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=2000,
    C=1.0
)

lr_l1.fit(X_train_nf, y_train_nf)
y_pred_proba = lr_l1.predict_proba(X_valid_nf)[:, 1]

gini_l1 = gini_sklearn(y_valid_nf, y_pred_proba)
print("Logistic Regression с L1 Gini:", gini_l1)


Logistic Regression с L1 Gini: 0.3224017308422893


Лучший - с регуляризацией L1

## 8. Hyperparameters Optimization

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, make_scorer

C_values = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]

best_gini = -np.inf
best_C = None
best_model = None

for C in C_values:
    lr = LogisticRegression(
        penalty='l1',
        solver='saga',
        C=C,
        max_iter=2000,
        random_state=42
    )

    lr.fit(X_train, y_train)

    y_pred_valid = lr.predict_proba(X_valid)[:, 1]
    gini_valid = gini_sklearn(y_valid, y_pred_valid)

    print(f"C={C:<5} | Gini (valid) = {gini_valid:.5f}")

    if gini_valid > best_gini:
        best_gini = gini_valid
        best_C = C
        best_model = lr


print("\nЛучший C:", best_C)
print("Лучший Gini на валидации:", best_gini)


C=0.01  | Gini (valid) = 0.30548
C=0.05  | Gini (valid) = 0.32113
C=0.1   | Gini (valid) = 0.32885
C=0.5   | Gini (valid) = 0.33118
C=1     | Gini (valid) = 0.32212
C=2     | Gini (valid) = 0.31192
C=5     | Gini (valid) = 0.27732
C=10    | Gini (valid) = 0.24484

Лучший C: 0.5
Лучший Gini на валидации: 0.33118055036168537


Главный параметр C - обратная сила регуляризации

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

best_gini = -1
best_k = None

for k in [3,5,7,10,15,20]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    y_pred_valid = knn.predict_proba(X_valid)[:,1]
    gini = gini_sklearn(y_valid, y_pred_valid)
    
    print(f"n_neighbors={k}, Gini на валидации={gini:.4f}")
    7
    if gini > best_gini:
        best_gini = gini
        best_k = k

print("Лучшее n_neighbors:", best_k)
print("Лучший Gini на валидации:", best_gini)


n_neighbors=3, Gini на валидации=0.1354
n_neighbors=5, Gini на валидации=0.1725
n_neighbors=7, Gini на валидации=0.1894
n_neighbors=10, Gini на валидации=0.2148
n_neighbors=15, Gini на валидации=0.2402
n_neighbors=20, Gini на валидации=0.2587
Лучшее n_neighbors: 20
Лучший Gini на валидации: 0.25869828061024136


Главный параметр кол-во соседей

In [ ]:
from sklearn.naive_bayes import GaussianNB

best_gini = -1
best_var = None

for var in [1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e4, 1e3, 1e2, ]:
    nb = GaussianNB(var_smoothing=var)
    nb.fit(X_train.toarray(), y_train)
    
    y_pred_valid = nb.predict_proba(X_valid.toarray())[:,1]
    gini = gini_sklearn(y_valid, y_pred_valid)
    
    print(f"var_smoothing={var}, Gini на валидации={gini:.4f}")
    
    if gini > best_gini:
        best_gini = gini
        best_var = var

print("Лучший var_smoothing:", best_var)
print("Лучший Gini на валидации:", best_gini)


var_smoothing=1e-09, Gini на валидации=0.0521
var_smoothing=1e-08, Gini на валидации=0.0521
var_smoothing=1e-07, Gini на валидации=0.0521
var_smoothing=1e-06, Gini на валидации=0.0530
var_smoothing=1e-05, Gini на валидации=0.0567
var_smoothing=10000.0, Gini на валидации=0.2743
var_smoothing=1000.0, Gini на валидации=0.2743
var_smoothing=100.0, Gini на валидации=0.2744
Лучший var_smoothing: 100.0
Лучший Gini на валидации: 0.2743571178657489


In [ ]:
from sklearn.svm import SVC

best_gini = -1
best_params = {}

for C in [0.01,0.1,1,10]:
    for gamma in ['scale']:
        svc = SVC(C=C, gamma=gamma, probability=True, random_state=42)
        svc.fit(X_train, y_train)
        
        y_pred_valid = svc.predict_proba(X_valid)[:,1]
        gini = gini_sklearn(y_valid, y_pred_valid)
        
        print(f"C={C}, gamma={gamma}, Gini={gini:.4f}")
        
        if gini > best_gini:
            best_gini = gini
            best_params = {'C': C, 'gamma': gamma}

print("Лучшие параметры SVM:", best_params)
print("Лучший Gini на валидации:", best_gini)


C=0.01, gamma=scale, Gini=0.2114
C=0.01, gamma=auto, Gini=0.0200
C=0.1, gamma=scale, Gini=0.2260
C=0.1, gamma=auto, Gini=0.0426
C=1, gamma=scale, Gini=0.2274
C=1, gamma=auto, Gini=0.1489
C=10, gamma=scale, Gini=0.2330
C=10, gamma=auto, Gini=0.1350
Лучшие параметры SVM: {'C': 10, 'gamma': 'scale'}
Лучший Gini на валидации: 0.23297101541822496


## 10. Best model gini on all three datasets
 

In [26]:
X_test = test_df.drop(columns=[target, date])
y_test = test_df[target]
X_test = preprocessor.transform(X_test)

best_lr = LogisticRegression(
        penalty='l1',
        solver='saga',
        C=0.5,
        max_iter=2000,
        random_state=42
    )

best_lr.fit(X_train, y_train)

y_pred_train = best_lr.predict_proba(X_train)[:, 1]
gini_train = gini_sklearn(y_train, y_pred_train)

y_pred_valid = best_lr.predict_proba(X_valid)[:, 1]
gini_valid = gini_sklearn(y_valid, y_pred_valid)

y_pred_test = best_lr.predict_proba(X_test)[:, 1]
gini_test = gini_sklearn(y_test, y_pred_test)

print("Gini на обучающем наборе:", gini_train)
print("Gini на валидационном наборе:", gini_valid)
print("Gini на тестовом наборе:", gini_test)


Gini на обучающем наборе: 0.45605146474486724
Gini на валидационном наборе: 0.33118055036168537
Gini на тестовом наборе: 0.3524571709399571


Модель переобучилась, это видно по сильной разнице между обучающим и остальными данными. 

In [21]:
def recall(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fn) if (tp + fn) > 0 else 0

In [22]:
def precision(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

In [23]:
def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

In [24]:
def auc_pr(y_true, y_score):
    order = np.argsort(-y_score)
    y_true = y_true[order]

    tp = 0
    fp = 0
    fn = np.sum(y_true)

    precisions = []
    recalls = []

    for y in y_true:
        if y == 1:
            tp += 1
            fn -= 1
        else:
            fp += 1

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0

        precisions.append(p)
        recalls.append(r)
        
    return np.trapz(precisions, recalls)


In [ ]:
models_predictions = {
    'MyLogisticRegression': MyLogisticRegression().fit(X_train, y_train).predict_proba(X_test)[:, 1],
    'MyGaussianNB': GaussianNB(var_smoothing=100).fit(X_train.toarray(), y_train).predict_proba(X_test.toarray())[:, 1],
    'MyKNN': MyKNN(n_neighbors=20).fit(X_train, y_train).predict_proba(X_test)[:,1],
    'LogisticRegression_L1': best_lr.predict_proba(X_test)[:, 1],
}

results = []
for name, pred in models_predictions.items():
    precision, recall, _ = precision_recall_curve(y_test.values, pred)
    auc_pr_score = auc(recall, precision)
    results.append({'Model': name, 'AUC-PR': auc_pr_score})

results_df = pd.DataFrame(results).sort_values('AUC-PR', ascending=False)
print(results_df)

                   Model    AUC-PR
3  LogisticRegression_L1  0.228308
2                  MyKNN  0.199511
0   MyLogisticRegression  0.199374
1           MyGaussianNB  0.196073


Для задачи выявления некачественных автомобилей с жёсткой маркировкой я предлагаю комбинированный подход: использовать механизм голосования экспертов с порогом 75% для получения качественной разметки данных, а для оценки модели применять F1-Score, который оптимально балансирует точность (минимизация ложных обвинений исправных авто) и полноту (выявление максимального числа реальных некачественных), при этом в эксплуатации спорные случаи стоит направлять на дополнительную проверку.